# Notebook 01 — Data Acquisition and Corpus Analysis
**CliniVer** · Stage A (controlled sample, parallel acquisition) · Experiment `EXP-01-001`

| | |
|---|---|
| **Objective** | Build and validate a multi-source acquisition pipeline for the cardiometabolic pharmacotherapy evidence corpus (ADR-0001, ADR-0002), and measure the corpus before scaling |
| **Research question** | RQ0 (partial): *What does an openly licensed, multi-source corpus for this domain look like (size, evidence types, dates, metadata quality, normalization coverage)?* Claim-level coverage is measured later, in NB05/NB08 |
| **Hypothesis** | None. This notebook is descriptive. It produces the measurements that later hypotheses depend on |
| **Stage A scope** | A **controlled sample** to validate parsers, schemas and data quality, plus **population counts** (count-only queries) to size Stage B. Scaling happens only after review |

> **Sampling caveat.** The PubMed sample uses NCBI *Best Match* ordering (top 100 per condition). It is a convenience sample for validating parsers, **not** a representative sample. Population-level statements in this notebook use the count-only queries, which cover the full query results.

## 1. Background and methodology

### 1.1 Sources, roles and terms (verified 2026-11-19)
| Source | Role | Access | Terms / limits used |
|---|---|---|---|
| PubMed (E-utilities) | evidence | keyless | ≤ 3 req/s without an API key, counted on arrival. Now 2 req/s with ≤ 2 in flight (the first parallel run used 2.5 req/s and got HTTP 429s; see EXP-01-002); `tool=cliniver`; abstracts may be under publisher copyright, so they are stored locally and **not redistributed** |
| PubTator 3 | graph input (annotations) | keyless | ≤ 3 req/s used |
| Europe PMC OA | evidence (full text) | keyless | Only CC-licensed OA articles; license stored per document |
| openFDA drug labels | regulatory evidence | keyless | CC0 (open.fda.gov/license); ~1,000 req/day keyless |
| ClinicalTrials.gov v2 | trial registry | keyless | No official limit published; paced at ~46 req/min |
| RxNorm / RxClass | normalization (not evidence) | keyless | ≤ 20 req/s per NLM; drug–drug interaction API **discontinued 2024-01-02** |
| PubChem | normalization (not evidence) | keyless | ≤ 5 req/s |
| WHO GHO (OData) | population statistics | keyless | CC BY-NC-SA 3.0 IGO; legacy endpoint live on 2026-11-19 despite announced rework |

**NCBI API key:** not available to the project owner, so PubMed runs at the keyless rate. This affects only speed (≤ 200 records per efetch request), not the data.

### 1.2 Pipeline
`configs/data.yaml` → source clients (`cliniver.ingestion.*`; rate-limited, retrying) → **raw archive** (`data/raw/<source>/<snapshot>/`, unchanged bytes + metadata sidecar with SHA-256) → parsers → unified `Document` / `Concept` / PubTator annotation records (`cliniver.schemas`) → `data/interim/<snapshot>/stageA/*.jsonl` → quality analysis (`cliniver.data.quality`).

Re-running this notebook re-uses the archived raw responses (no new API calls), so results are reproducible for a given snapshot.

## 2. Environment and configuration

In [ ]:
import json, time
import xml.etree.ElementTree as ET
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import date, datetime, timezone

import matplotlib.pyplot as plt
import pandas as pd

from cliniver.config import load_project_config
from cliniver.config.data_config import load_data_config
from cliniver.data.corpus_io import write_jsonl
from cliniver.data.quality import completeness, docs_to_frame, merge_duplicate_ids, near_duplicate_clusters, near_duplicate_pairs
from cliniver.data.report import composition, evidence_stratum, pubtator_coverage, quality_flags
from cliniver.ingestion.clinicaltrials import ClinicalTrialsClient, parse_study
from cliniver.ingestion.europepmc import EuropePMCClient, parse_jats
from cliniver.ingestion.http import FetchError, RawStore, parallel_map
from cliniver.ingestion.normalization import PubChemClient, RxNavClient
from cliniver.ingestion.openfda import OpenFDAClient, parse_label
from cliniver.ingestion.pubmed import PubMedClient, parse_pubmed_xml
from cliniver.ingestion.pubtator import PubTatorClient, parse_bioc
from cliniver.ingestion.who_gho import WhoGhoClient, textualize
from cliniver.utils import collect_hardware_info, set_seed, setup_logging
from cliniver.utils.experiment import Experiment

pcfg = load_project_config()
dcfg = load_data_config()
setup_logging(pcfg.logging.level, pcfg.paths.absolute("logs"), run_name="nb01")
SEED = set_seed(pcfg.reproducibility.seed)
SNAP = dcfg.snapshot
RAW = pcfg.paths.absolute("data_raw")
OUT = pcfg.paths.absolute("data_interim") / SNAP / "stageA"
OUT.mkdir(parents=True, exist_ok=True)

exp = Experiment(
    exp_id="EXP-01-001", title="Stage A multi-source acquisition and corpus quality analysis",
    research_question="RQ0 (partial): corpus composition, quality and normalization coverage",
    config=dcfg.model_dump(mode="json"), seed=SEED,
    packages=["torch", "pandas", "requests", "pydantic", "matplotlib", "cliniver"])
store = lambda src: RawStore(RAW, src, SNAP)
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
print("snapshot:", SNAP, "| stage:", dcfg.stage, "| seed:", SEED)
print("hardware:", {k: v for k, v in collect_hardware_info().to_dict().items() if k in ("python_version", "platform", "ram_total_gb")})

## 3. Acquisition (parallel)

**Concurrency design.**
- **Across providers:** each source runs as an independent job, up to `concurrency.sources_in_parallel` at once. Providers have independent rate limits, so they never wait on each other.
- **Within a provider:** up to `workers_per_source` requests can be in flight at once. Request **start times** are still spaced by that provider's `min_interval_s` (a thread-safe limiter shared by every thread using the client), so published limits are never exceeded. Only network latency overlaps.
- **Shared limits:** PubMed population counts and the PubMed sample are separate jobs that share one PubMed client, so their *combined* rate stays at 2.5 req/s.

Dependency: PubTator needs the PubMed sample PMIDs, so the two run in sequence inside one job.

**Population counts** (part of the `pubmed_counts` job) download **no records**. They return NCBI's hit count per query, which gives an unbiased picture of the *full* literature per condition, by publication type and by decade. Stage B is sized from these counts.

In [ ]:
W = dcfg.concurrency.workers_per_source
pm = PubMedClient(dcfg.pubmed, store("pubmed"), W)
pt = PubTatorClient(dcfg.pubtator, store("pubtator"), W)
fda = OpenFDAClient(dcfg.openfda, store("openfda"), W)
ct = ClinicalTrialsClient(dcfg.clinicaltrials, store("clinicaltrials"), W)
ep = EuropePMCClient(dcfg.europepmc, store("europepmc"), W)
rx, pc = RxNavClient(dcfg.rxnav, store("rxnav"), W), PubChemClient(dcfg.pubchem, store("pubchem"), W)
who = WhoGhoClient(dcfg.who_gho, store("who_gho"), W)
CONDS = list(dcfg.pubmed.conditions)


def acquire_pubmed_counts():
    qs = []
    for c in CONDS:
        qs.append(("total", c, None, (pm.query(c), f"{c}_total")))
        qs += [("type", c, t, (pm.query(c, f"AND {q}"), f"{c}_type_{t}")) for t, q in dcfg.pubmed.publication_type_strata.items()]
        qs += [("decade", c, f"{a}-{b}", (pm.query(c, f'AND ("{a}/01/01"[dp] : "{b}/12/31"[dp])'), f"{c}_dec_{a}"))
               for a, b in dcfg.pubmed.decades]
    out = {"total": {}, "by_type": defaultdict(dict), "by_decade": defaultdict(dict)}
    for (kind, c, k, _), v in zip(qs, pm.count_many([q[3] for q in qs])):
        if kind == "total": out["total"][c] = v
        else: out["by_type" if kind == "type" else "by_decade"][c][k] = v
    return out


def acquire_pubmed_sample_and_pubtator():
    ids = dict(zip(CONDS, parallel_map(lambda c: pm.search_ids(pm.query(c), dcfg.pubmed.sample_per_condition, f"{c}_sample"), CONDS, W)))
    xml_per_cond = parallel_map(lambda c: pm.fetch_xml(ids[c], f"{c}_sample"), CONDS, W)
    docs = [d for c, xmls in zip(CONDS, xml_per_cond) for x in xmls for d in parse_pubmed_xml(x, snapshot=SNAP, condition=c)]
    docs, dup = merge_duplicate_ids(docs)
    pmids = sorted({d.source_id for d in docs})
    bioc = pt.export(pmids, "stageA")
    ents, rels = [], []
    for b in bioc:
        e, r = parse_bioc(b); ents += e; rels += r
    return {"ids": ids, "docs": docs, "dup": dup, "pmids": pmids, "bioc": bioc, "entities": ents, "relations": rels}


def acquire_openfda():
    pairs = [(c, d) for c, ds in dcfg.drugs.items() for d in ds[: dcfg.openfda.sample_drugs_per_condition]]
    labels = parallel_map(lambda cd: fda.labels_for(cd[1], dcfg.openfda.labels_per_drug), pairs, W)
    docs = [parse_label(l, sections=dcfg.openfda.sections, snapshot=SNAP, query_drug=d, condition=c)
            for (c, d), ls in zip(pairs, labels) for l in ls]
    return {"docs": docs, "missing": [d for (c, d), ls in zip(pairs, labels) if not ls]}


def acquire_clinicaltrials():
    res = parallel_map(lambda c: ct.search(c, dcfg.clinicaltrials.sample_per_condition), CONDS, W)
    docs = [parse_study(s, snapshot=SNAP, condition=c) for c, (studies, _) in zip(CONDS, res) for s in studies]
    docs, dup = merge_duplicate_ids(docs)
    return {"docs": docs, "dup": dup, "totals": {c: t for c, (_, t) in zip(CONDS, res)}}


def acquire_europepmc():
    res = parallel_map(lambda c: ep.search_oa(c, dcfg.europepmc.sample_per_condition), CONDS, W)
    metas = [(c, m) for c, (results, _) in zip(CONDS, res) for m in results]

    def one(cm):
        c, meta = cm
        try:
            return parse_jats(ep.full_text_xml(meta["pmcid"]), meta, snapshot=SNAP, condition=c), None
        except (FetchError, KeyError) as exc:
            return None, (meta.get("pmcid"), str(exc)[:120])

    out = parallel_map(one, metas, W)
    return {"docs": [d for d, _ in out if d], "fail": [e for _, e in out if e], "hits": {c: h for c, (_, h) in zip(CONDS, res)}}


def acquire_normalization():
    def one(cd):
        c, drug = cd
        return c, drug, rx.concepts_for(drug, snapshot=SNAP), pc.concept_for(drug, snapshot=SNAP)
    return parallel_map(one, dcfg.all_drugs(), W)


def acquire_who():
    found = parallel_map(who.find_indicators, dcfg.who_gho.indicator_keywords, W)
    indicators = {i["IndicatorCode"]: i["IndicatorName"] for lst in found for i in lst}
    # Stage A: values only for age-standardized prevalence indicators, to validate the textualizer
    selected = {k: v for k, v in indicators.items() if "age-standardized" in v.lower()}
    rows = parallel_map(who.values, list(selected), W)
    docs = [textualize(r, name, snapshot=SNAP) for (code_, name), rs in zip(selected.items(), rows) for r in rs]
    return {"indicators": indicators, "selected": selected, "docs": docs}


JOBS = {"pubmed_counts": acquire_pubmed_counts, "pubmed_sample+pubtator": acquire_pubmed_sample_and_pubtator,
        "openfda": acquire_openfda, "clinicaltrials": acquire_clinicaltrials, "europepmc_oa": acquire_europepmc,
        "rxnav+pubchem": acquire_normalization, "who_gho": acquire_who}


def _timed(name, fn):
    t0 = time.perf_counter(); res = fn(); return name, res, time.perf_counter() - t0


t_all = time.perf_counter()
R, job_seconds = {}, {}
with ThreadPoolExecutor(max_workers=dcfg.concurrency.sources_in_parallel) as pool:
    for fut in as_completed([pool.submit(_timed, n, f) for n, f in JOBS.items()]):
        name, res, dt = fut.result()  # re-raises any job's exception: failures are never silent
        R[name], job_seconds[name] = res, dt
wall = time.perf_counter() - t_all
print(f"parallel acquisition wall time: {wall:.1f}s | sum of job times: {sum(job_seconds.values()):.1f}s")

### 3.1 Acquisition timing and comparison with the sequential baseline
Both timings are **measured from the raw archive**: the span between the first and last `retrieved_at` timestamps written by `RawStore` (1-second resolution).
- Snapshot `2026-11-19`: the earlier fully sequential implementation.
- The current snapshot: this parallel run.

Both runs fetched fresh from the APIs (no cache hits), so the comparison is like-for-like. Responses per source can differ slightly between days; request counts are shown alongside.

In [ ]:
def archive_span(snapshot):
    stamps, n = [], Counter()
    for meta in RAW.glob(f"*/{snapshot}/*.meta.json"):
        m = json.loads(meta.read_text(encoding="utf-8"))
        stamps.append(datetime.fromisoformat(m["retrieved_at"])); n[meta.parent.parent.name] += 1
    return ((max(stamps) - min(stamps)).total_seconds() if stamps else None), sum(n.values())

clients = {"pubmed": pm, "pubtator": pt, "openfda": fda, "clinicaltrials": ct, "europepmc": ep, "rxnav": rx, "pubchem": pc, "who_gho": who}
timing = pd.DataFrame({"seconds": pd.Series(job_seconds).round(1)}).sort_values("seconds", ascending=False)
req = pd.DataFrame({k: {"requests": c.http.n_requests, "retries": c.http.n_retries, "cache_hits": c.http.n_cache_hits,
                        "min_interval_s": c.http.min_interval_s,
                        "floor_s_if_rate_bound": round(c.http.n_requests * c.http.min_interval_s, 1)} for k, c in clients.items()}).T
seq_span, seq_files = archive_span("2026-11-19")
par_span, par_files = archive_span(SNAP)
cmp_ = {"sequential_snapshot": "2026-11-19", "sequential_span_s": seq_span, "sequential_raw_files": seq_files,
        "parallel_snapshot": SNAP, "parallel_span_s": par_span, "parallel_raw_files": par_files,
        "parallel_wall_s_in_notebook": round(wall, 1),
        "speedup_span_ratio": round(seq_span / par_span, 2) if seq_span and par_span else None}
REPLAY = int(req["requests"].sum()) == 0  # every response came from the raw archive
cmp_["note"] = ("REPLAY of the raw archive: job seconds below measure parsing only, not acquisition. Live acquisition timing: "
               "artifacts/experiments/EXP-01-001/live_acquisition_run_2026-11-20_*.") if REPLAY else "live acquisition run"
exp.log_result("replay_timing" if REPLAY else "acquisition_timing", {"jobs_s": job_seconds, "wall_s": wall, "requests": req.to_dict(orient="index")})
exp.log_result("acquisition_span_comparison", cmp_)
exp.save_table("acquisition_timing_jobs", timing); exp.save_table("acquisition_requests", req)
print(json.dumps(cmp_, indent=2))
display(timing); display(req)

In [ ]:
# Unpack job results for the analysis sections
pop = R["pubmed_counts"]
pop_total = pop["total"]
pop_type_df = pd.DataFrame(pop["by_type"]).T.loc[CONDS]
pop_type_df.insert(0, "TOTAL", pd.Series(pop_total))
pop_decade_df = pd.DataFrame(pop["by_decade"]).T.loc[CONDS]
exp.save_table("pubmed_population_by_type", pop_type_df); exp.save_table("pubmed_population_by_decade", pop_decade_df)
exp.log_result("pubmed_population_total_per_condition", pop_total)

S = R["pubmed_sample+pubtator"]
pm_ids, pm_docs, pm_dup_ids, all_pmids, bioc, entities, relations = (S[k] for k in ("ids", "docs", "dup", "pmids", "bioc", "entities", "relations"))
annotated = {e.pmid for e in entities}
fda_docs, fda_missing = R["openfda"]["docs"], R["openfda"]["missing"]
ct_docs, ct_dup, ct_totals = R["clinicaltrials"]["docs"], R["clinicaltrials"]["dup"], R["clinicaltrials"]["totals"]
ep_docs, ep_fail, ep_hits = R["europepmc_oa"]["docs"], R["europepmc_oa"]["fail"], R["europepmc_oa"]["hits"]
indicators, selected, who_docs = R["who_gho"]["indicators"], R["who_gho"]["selected"], R["who_gho"]["docs"]
exp.log_result("clinicaltrials_population_completed_drug_trials", ct_totals)
exp.log_result("europepmc_oa_population_hits", ep_hits)
print("Totals are per condition; documents can belong to more than one condition.")
pop_type_df

In [ ]:
pop_decade_df

## 4. Per-source acquisition results

### 4.1 PubMed sample (top 100 per condition, Best Match) and 4.2 PubTator 3 annotations

In [ ]:
requested = sum(len(v) for v in pm_ids.values())
print(f"requested IDs: {requested} | unique parsed docs: {len(pm_docs)} | same PMID under >1 condition: {pm_dup_ids}")
print(f"PMIDs sent to PubTator: {len(all_pmids)} | returned: {len(bioc)} | with >=1 entity: {len(annotated)}")
print(f"entity mentions: {len(entities)} | relations: {len(relations)}")

### 4.3 openFDA drug labels (2 drugs per condition × up to 3 labels)

In [ ]:
print(f"labels: {len(fda_docs)} | drugs with no exact-name label: {fda_missing}")

### 4.4 ClinicalTrials.gov (completed, drug-intervention trials; 20 per condition)

In [ ]:
print("population (completed drug trials):", ct_totals)
print(f"sample docs: {len(ct_docs)} (cross-condition duplicates merged: {ct_dup})")

### 4.5 Europe PMC open-access full text (license-filtered; 2 per condition)

In [ ]:
print("OA full-text population hits:", ep_hits)
print(f"full texts parsed: {len(ep_docs)} | failures: {ep_fail}")

### 4.6 Drug normalization: RxNorm / RxClass and PubChem (all 49 in-scope drugs)

In [ ]:
concepts, norm_rows = [], []
for c, drug, rxc, pcc in R["rxnav+pubchem"]:
    concepts += rxc + ([pcc] if pcc else [])
    drug_c = next((x for x in rxc if x.kind == "drug"), None)
    norm_rows.append({"condition": c, "drug": drug, "rxcui": drug_c.concept_id if drug_c else None,
                      "n_brands": len(drug_c.synonyms) if drug_c else 0,
                      "n_classes": sum(x.kind == "drug_class" for x in rxc),
                      "pubchem_cid": pcc.concept_id if pcc else None,
                      "n_pubchem_synonyms_total": pcc.extra["n_synonyms_total"] if pcc else 0})
norm_df = pd.DataFrame(norm_rows)
# Classes repeat across drugs: keep one record per concept_id with the union of member drugs
uniq = {}
for x in concepts:
    if x.concept_id in uniq and x.kind == "drug_class":
        prev = uniq[x.concept_id]
        members = set(prev.extra.get("members", [prev.extra["member_rxcui"]])) | {x.extra["member_rxcui"]}
        uniq[x.concept_id] = prev.model_copy(update={"extra": {**prev.extra, "members": sorted(members)}})
    else:
        uniq.setdefault(x.concept_id, x)
concepts = list(uniq.values())
exp.save_table("normalization_coverage", norm_df)
print(f"concept records: {len(concepts)}  ({Counter(x.kind for x in concepts)})")
norm_df

### 4.7 WHO GHO population statistics (indicator discovery + global/regional values)

In [ ]:
print(f"{len(indicators)} indicators matched keywords | selected (age-standardized): {len(selected)} | statistic documents: {len(who_docs)}")
pd.Series(selected, name="indicator").to_frame()

## 5. Persist normalized records

In [ ]:
all_docs = pm_docs + fda_docs + ct_docs + ep_docs + who_docs
counts = {
    "documents": write_jsonl(all_docs, OUT / "documents.jsonl"),
    "concepts": write_jsonl(concepts, OUT / "concepts.jsonl"),
    "pubtator_entities": write_jsonl(entities, OUT / "pubtator_entities.jsonl"),
    "pubtator_relations": write_jsonl(relations, OUT / "pubtator_relations.jsonl"),
}
manifest = {"snapshot": SNAP, "stage": "A", "created_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
            "counts": counts, "by_source": dict(Counter(d.source for d in all_docs))}
(OUT / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
exp.log_result("record_counts", manifest)
manifest

## 6. Evaluation: corpus quality

### 6.1 Composition and text length per source

In [ ]:
df = docs_to_frame(all_docs)
comp = composition(df)
exp.save_table("composition", comp); exp.log_result("composition", comp.to_dict(orient="index"))
comp

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
order = comp.index.tolist()
ax.boxplot([df.loc[df.source == s, "n_words"].clip(lower=1) for s in order], tick_labels=order, showfliers=True)
ax.set_yscale("log"); ax.set_ylabel("words per document (log)"); ax.set_title("Document length by source (Stage A sample)")
plt.xticks(rotation=20); exp.save_figure("length_by_source", fig); plt.show()

### 6.2 Metadata completeness (fraction of documents with each field)

In [ ]:
compl = completeness(all_docs)
exp.save_table("completeness", compl); exp.log_result("completeness", compl.round(3).to_dict(orient="index"))
fig, ax = plt.subplots(figsize=(9, 3))
im = ax.imshow(compl.values, vmin=0, vmax=1, cmap="viridis", aspect="auto")
ax.set_xticks(range(len(compl.columns)), compl.columns, rotation=40, ha="right"); ax.set_yticks(range(len(compl.index)), compl.index)
for i in range(compl.shape[0]):
    for j in range(compl.shape[1]):
        ax.text(j, i, f"{compl.values[i, j]:.2f}", ha="center", va="center", color="white" if compl.values[i, j] < .6 else "black", fontsize=8)
fig.colorbar(im, ax=ax, fraction=.02); ax.set_title("Field completeness by source")
exp.save_figure("completeness", fig); plt.show()
compl.round(3)

### 6.3 Duplicates: exact, cross-source and near-duplicate

In [ ]:
pm_pmids = {d.source_id for d in pm_docs}
cross = [d.doc_id for d in ep_docs if d.identifiers.get("pmid") in pm_pmids]
dois = [d.identifiers.get("doi") for d in pm_docs if d.identifiers.get("doi")]
label_nd = near_duplicate_pairs(fda_docs, group_key=lambda d: d.extra["query_drug"], threshold=0.8)
dup = {"pubmed_same_pmid_multiple_conditions": pm_dup_ids, "clinicaltrials_same_nct_multiple_conditions": ct_dup,
       "pubmed_duplicate_dois": len(dois) - len(set(dois)), "europepmc_oa_also_in_pubmed_sample": len(cross),
       "openfda_label_pairs_same_drug": len(label_nd), "openfda_near_dup_pairs_jaccard>=0.8": int(label_nd.near_dup.sum()) if len(label_nd) else 0}
exp.log_result("duplicates", dup); exp.save_table("openfda_label_similarity", label_nd)
print(json.dumps(dup, indent=2))
label_nd.groupby("group").jaccard.describe()[["count", "mean", "min", "max"]].round(3) if len(label_nd) else "no label pairs"

### 6.4 Evidence types: PubMed sample vs. population

In [ ]:
stratum = evidence_stratum  # shared definition (cliniver.data.report)
pm_df = df[df.source == "pubmed"].copy()
pm_df["stratum"] = pm_df.publication_types.map(stratum)
sample_share = pm_df.stratum.value_counts(normalize=True).rename("sample_share")
pop_sum = pop_type_df.drop(columns="TOTAL").sum()
pop_share = (pop_sum / pop_type_df["TOTAL"].sum()).rename("population_share_upper_bound")
evid = pd.concat([sample_share, pop_share], axis=1).fillna(0).round(3)
exp.save_table("evidence_type_sample_vs_population", evid)
print("Population shares: strata may overlap (e.g. an RCT also tagged Review), so the shares are upper bounds and need not sum to 1.")
evid

### 6.5 Temporal distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
pop_decade_df.T.plot(kind="bar", ax=axes[0]); axes[0].set_title("PubMed population by decade (count queries)"); axes[0].set_ylabel("articles")
for s, g in df[df.year.notna()].groupby("source"):
    axes[1].hist(g.year, bins=range(1990, 2028), alpha=.55, label=s)
axes[1].set_title("Stage A sample: document year by source"); axes[1].legend(fontsize=7)
plt.tight_layout(); exp.save_figure("temporal", fig); plt.show()
yr = df.groupby("source").year.describe()[["count", "min", "50%", "max"]]
exp.save_table("year_by_source", yr); yr

### 6.6 Document-quality flags

In [ ]:
q = quality_flags(all_docs)
exp.log_result("quality_flags", q); pd.Series(q, name="value").to_frame()

### 6.7 Coverage

**(a) Drug normalization coverage.** Can every in-scope drug be linked to RxNorm, RxClass and PubChem?

In [ ]:
cov = {"drugs": len(norm_df), "rxnorm_found": int(norm_df.rxcui.notna().sum()), "with_rxclass": int((norm_df.n_classes > 0).sum()),
       "pubchem_found": int(norm_df.pubchem_cid.notna().sum()),
       "not_in_pubchem": norm_df.loc[norm_df.pubchem_cid.isna(), "drug"].tolist(),
       "not_in_rxnorm": norm_df.loc[norm_df.rxcui.isna(), "drug"].tolist()}
exp.log_result("normalization_coverage", cov); print(json.dumps(cov, indent=2))

**(b) PubTator entity coverage of the PubMed sample.** This tests whether graph construction (ADR-0003) can rely on precomputed annotations: what share of documents carry at least one Chemical and one Disease annotation, and how many mention an in-scope drug (exact name match against generic and RxNorm brand names; a lower bound)?

In [ ]:
drug_names = {d.lower() for _, d in dcfg.all_drugs()} | {s.lower() for x in concepts if x.source == "rxnorm" for s in x.synonyms}
pt_cov = pubtator_coverage(entities, relations, pm_docs, drug_names)
exp.log_result("pubtator_coverage", pt_cov); print(json.dumps(pt_cov, indent=2))

## 7. Stage B sizing (projection from measured counts)
This is a projection, not a measurement. It assumes the measured Stage A request behaviour (≤ 200 records per efetch at 2.5 req/s) and the measured median document size.

In [ ]:
union_upper = int(pop_type_df["TOTAL"].sum())  # conditions overlap, so the union is smaller
pm_bytes_per_doc = (OUT / "documents.jsonl").stat().st_size / max(len(all_docs), 1)
proj = {
    "pubmed_population_sum_over_conditions_upper_bound": union_upper,
    "efetch_requests_needed": -(-union_upper // dcfg.pubmed.efetch_batch),
    "est_minutes_pubmed_at_configured_rate": round((union_upper / dcfg.pubmed.efetch_batch + union_upper / 9999) * dcfg.pubmed.min_interval_s / 60, 1),
    "est_minutes_pubtator_at_configured_rate": round(union_upper / dcfg.pubtator.batch * dcfg.pubtator.min_interval_s / 60, 1),
    "pubtator_requests_needed": -(-union_upper // dcfg.pubtator.batch),
    "est_normalized_jsonl_mb": round(union_upper * pm_bytes_per_doc / 1e6, 1),
}
exp.log_result("stageB_projection", proj); print(json.dumps(proj, indent=2))

## 8. Error analysis: failures and edge cases encountered

In [ ]:
errors = {"openfda_no_label_for_name": fda_missing, "europepmc_fulltext_failures": ep_fail,
          "pubmed_ids_not_parsed": sorted(set(sum(pm_ids.values(), [])) - {d.source_id for d in pm_docs}),
          "pubtator_pmids_missing": sorted(set(all_pmids) - annotated),
          "http": {name: {"requests": c.http.n_requests, "retries": c.http.n_retries}
                   for name, c in [("pubmed", pm), ("pubtator", pt), ("openfda", fda), ("clinicaltrials", ct),
                                   ("europepmc", ep), ("rxnav", rx), ("pubchem", pc), ("who_gho", who)]}}
exp.log_result("errors", errors); print(json.dumps(errors, indent=2))

In [ ]:
# Inspect one record per source so the schema can be checked by eye
for src in ["pubmed", "openfda_label", "clinicaltrials", "europepmc_oa", "who_gho"]:
    d = next(x for x in all_docs if x.source == src)
    print(f"--- {src}: {d.doc_id} | {d.pub_date} ({d.date_precision}) | license: {d.license}")
    print("title:", d.title[:110]); print("sections:", [s.name for s in d.sections][:8]); print("text:", d.text[:240].replace("\n", " "), "\n")

In [ ]:
exp.finish()
print("results:", exp.dir)

## 9. Discussion
All numbers below are taken from the outputs above, which are written to `artifacts/experiments/EXP-01-001/`.

### 9.1 Population-level findings (count queries; these generalize)
| Condition | PubMed articles (MeSH `/drug therapy`, human, English, with abstract, 1990–) | Completed drug trials (ClinicalTrials.gov) |
|---|---|---|
| Type 2 diabetes | 40,779 | 4,177 |
| Hypertension | 40,306 | 2,971 |
| Heart failure | 18,325 | 1,129 |
| AF / anticoagulation | 15,744 | 537 |
| Dyslipidaemia | 15,107 | 1,360 |

- **Size.** The per-condition sum (130,261) is an *upper bound* on unique articles, because conditions overlap. The full domain literature is small enough to acquire **completely**, so no sampling is needed (projection in §7: ~5.5 min of PubMed requests at the configured rate, ~318 MB of JSONL).
- **Evidence types** (shares are upper bounds because strata overlap): narrative reviews ≤ 25.8%, RCTs ≤ 17.9%, meta-analyses ≤ 4.8%, systematic reviews ≤ 4.0%, other clinical trials ≤ 4.2%, case reports ≤ 4.1%, observational ≤ 3.6%, guidelines ≤ 0.3%. High-tier designs (MA, SR, RCT) make up a substantial minority, which supports a `source_tier` confidence signal (NB07).
- **Time.** Type 2 diabetes literature is still growing (1,520 articles in the 1990s vs. 17,922 in 2020–2026). Hypertension peaked in 2010–2019 (13,480). Evidence recency therefore varies by topic, which supports per-evidence dating (ADR-0005).
- **Trial registry.** Only 27 of 97 sampled completed trials (28%) had posted results. Most registry records describe design, not outcomes, so they are kept as `trial_registry`, not as efficacy evidence.
- Europe PMC OA hit counts come from broader free-text queries and are **not comparable** with the MeSH-restricted PubMed counts.

### 9.2 Parser and schema validation (the purpose of the Stage A sample)
- All five document sources and both normalization sources produced valid records in the unified schema: 494/494 PubMed IDs parsed; 0 openFDA name misses; 0 Europe PMC full-text failures (after retries).
- **Two parser defects were found and fixed** (with regression tests):
  1. 10/494 (2.0%) PubMed records had their English abstract only in `<OtherAbstract Language="eng">`, which left their text empty. These are now recovered (`pubmed_empty_abstract = 0`).
  2. sacubitril, marketed only in combination, received no drug class because of the combination filter. It now falls back to combination classes, flagged with `via_combination`.
- **Truncation check.** Abstracts are not truncated. A known long RCT abstract (EMPA-REG OUTCOME, PMID 26378978) parses to all 297 words in 4 structured sections, matching NCBI's own text format.

### 9.3 The Best Match sample is not representative
Sample vs. population: RCTs 0.6% vs ≤ 17.9%; narrative reviews 48.6% vs ≤ 25.8%. The median sample abstract is 67 words, and 168/494 have fewer than 50 words. Spot checks show news items and short narrative reviews ranked highly. **No statistic from the PubMed sample should be generalized.** Stage B should acquire the full defined population (decision D-NB01-1).

### 9.4 Source-specific issues that need Stage B decisions
- **WHO GHO:** 3,444 statements (85% of Stage A documents) from just 4 indicators × 3 sex categories × years 1980–2022 × global/regional locations. Pooled with the other sources, they would dominate retrieval by count alone.
- **openFDA:** 15 of 30 same-drug label pairs are near-duplicates (word 5-gram Jaccard ≥ 0.8), mostly different manufacturers' labels for the same generic. 6 labels were combination products. Label dates are *revision* dates (all 2026), not dates of the underlying evidence.
- **PubTator 3:**
  - Coverage: 478/494 documents (96.8%) have annotations, and 16 PMIDs are missing (7 of them from 2024, consistent with processing lag). Disease annotations: 90% of documents. Chemical: 57%. Both: 52%.
  - Drug mentions: an exact-name match for an in-scope drug appears in only 101/494 documents. This is a lower bound (salts, classes and abbreviations are missed), but it shows that **normalization quality will limit the graph channel**.
  - Relation labels are coarse statistical types (Negative_Correlation 333 of 567, Positive_Correlation 106, Association 96). They are not "treats/causes" semantics, so **graph edges must never be read as evidence of an effect or its direction** (similarity ≠ verification).
- **Normalization:** RxNorm 49/49, RxClass 49/49 (1 via combination), PubChem 46/49. The three misses (dulaglutide, evolocumab, alirocumab) are biologics, which PubChem's compound database is not designed to hold. RxNorm should therefore be the normalization backbone, with PubChem as a supplementary synonym source for small molecules.
- **Europe PMC OA:** all 10 sampled full texts are from 2026 (ordering bias). Median length 4,299 words. 19 tables are not part of the extracted text, which NB02 must account for.
- **ClinicalTrials.gov:** 47/97 records have no detailed description, only a brief summary.

### 9.5 Acquisition engineering
- **Parallel acquisition is 4.15× faster.** The raw-archive span was 189 s, vs 784 s for the earlier sequential implementation, with the same 408 raw files. The remaining bottleneck is Europe PMC's full-text endpoint (60 s read timeouts, 3 retries); every other job finished in ≤ 44 s.
- **Rate-limit incident.** The first parallel run received **14 HTTP 429 responses from NCBI** at 2.5 req/s with 4 requests in flight. NCBI counts by arrival time, and network jitter bunched arrivals. All 14 succeeded on retry, so no data was lost, but the provider's limit had been exceeded. The fix (2 req/s, ≤ 2 in flight) was validated live in **EXP-01-002**: 70 fresh requests under the same two-job contention, **0 rejections**.
- **Reproducibility.** A replay of the snapshot makes **0 live requests**: 411 archive hits, including 3 archived "not found" answers, which were added after a replay exposed that 404s were not being archived.

## 10. Limitations
- The Stage A PubMed sample is unrepresentative (by design: it exists to validate parsers). Only the count-query results generalize.
- Population counts depend on MeSH indexing. Very recent articles (indexed automatically and later) may be under-counted. The `/drug therapy` subheading can miss pharmacology papers indexed only under the drug name.
- Publication-type shares are upper bounds because strata overlap.
- Some provider terms (openFDA keyless per-minute limit, ClinicalTrials.gov pacing) were verified from secondary sources because the official pages could not be retrieved programmatically. We use conservative rates.
- Europe PMC timing depends on server latency on the day of acquisition.
- Near-duplicate detection is pairwise (quadratic). That is adequate for 30 labels but not for Stage B (needs MinHash/LSH).

## 11. Conclusion and decisions

**Conclusion (Stage A).** The multi-source acquisition pipeline is validated:
- every source parses into the unified schema;
- the two parser defects found are fixed and covered by tests;
- acquisition is parallel and within every provider's limits;
- the raw archive reproduces the snapshot exactly.

The domain literature is small enough (≤ 130k PubMed articles) to acquire in full.

**Proposed decisions for Stage B** (to be confirmed by the project owner):

| ID | Decision | Rationale (evidence above) |
|---|---|---|
| D-NB01-1 | PubMed: acquire the **full defined population** (all five condition queries), no sampling | Best Match sample is heavily biased (§9.3). Full acquisition is cheap (§7) |
| D-NB01-2 | PubTator: annotations for all acquired PMIDs; record the missing ones | 96.8% coverage in Stage A |
| D-NB01-3 | openFDA: all 49 drugs, prescription labels; collapse near-duplicate clusters to the most recent label, keeping the others' set-ids as aliases | 15/30 near-duplicate pairs (§9.4) |
| D-NB01-4 | ClinicalTrials.gov: all completed drug trials for the five conditions; keep `has_results`; registry-only records stay `trial_registry` | 28% have posted results |
| D-NB01-5 | Europe PMC OA: a **bounded, year-stratified** long-document subset (~300–500 articles) for the NB02 chunking study, not the full OA set | Long-document study only; 2026-only ordering bias seen |
| D-NB01-6 | WHO GHO: keep all statements, role `statistics`; how (or whether) they enter retrieval is decided in NB05 (e.g., a separate index used only for epidemiological claims) | 85% of Stage A documents from 4 indicators (§9.4) |
| D-NB01-7 | Scale near-duplicate detection with MinHash/LSH | Pairwise approach does not scale |
| D-NB01-8 | RxNorm/RxClass is the normalization backbone; PubChem is supplementary | PubChem 46/49; biologics absent |

## 12. Next steps
1. Owner review of the Stage A findings and decisions D-NB01-1 … 8.
2. Stage B: scaled acquisition and full-corpus analysis (same sections 6–8, on the full corpus), with MinHash near-duplicate detection.
3. Then NB02 (document processing and chunking) on the long-document subset.

---
# Part B — Stage B: full corpus (Experiment `EXP-01-003`)

**Decisions adopted.** On 2026-11-20 the project owner delegated the Stage B decisions. The proposals from §11 were adopted unchanged, plus one safety rule (recorded in `docs/decisions/ADR-0008-corpus-composition.md`):

| ID | Decision as implemented |
|---|---|
| D-NB01-1 | PubMed: **full** population of the five condition queries, retrieved one publication year at a time (esearch returns ≤ 10k IDs per query; each slice is checked against its reported count) |
| D-NB01-2 | PubTator 3 annotations for every acquired PMID (parsed per batch) |
| D-NB01-3 | openFDA: all 49 drugs, **all** prescription labels (paged; the planned 100-label cap would have truncated 8 drugs, found in the smoke test); near-duplicate clusters collapsed to their most recent label |
| D-NB01-4 | ClinicalTrials.gov: all completed drug-intervention trials (paged; checked against `totalCount`) with trial → publication links |
| D-NB01-5 | Europe PMC OA: year-stratified long-document subset (5 conditions × 4 periods × 20) |
| D-NB01-6 | WHO GHO statements reused from Stage A (same snapshot); retrieval role decided in NB05 |
| D-NB01-7 | MinHash/LSH near-duplicate detection (128 permutations, 32 bands) with an exact Jaccard check |
| D-NB01-8 | RxNorm/RxClass is the normalization backbone; PubChem supplementary (Stage A concept records reused) |
| **Safety** | **Retracted publications are excluded from the evidence corpus** (listed, never used as evidence) |

Stage B raw responses go to `data/raw/<source>/<snapshot>/stageB/`, kept separate from the Stage A archive.

In [ ]:
import os
# SMOKE mode (env CLINIVER_NB01_SMOKE=1): a small end-to-end check of this Part B code before the long live
# run. Everything goes to separate *_smoke locations and a separate experiment id, so smoke data can never
# mix with the real Stage B archive (batch cache names would otherwise collide).
SMOKE = os.environ.get("CLINIVER_NB01_SMOKE") == "1"
cfgB = dcfg.stage_b if not SMOKE else dcfg.stage_b.model_copy(update={
    "pubmed_years": (2025, 2026), "openfda_max_labels_per_drug": 5, "clinicaltrials_page_size": 20,
    "europepmc_per_stratum": 1})
TAG = "stageB_smoke" if SMOKE else "stageB"
expB = Experiment(
    exp_id="EXP-01-003" + ("-smoke" if SMOKE else ""), title="Stage B full-corpus acquisition, deduplication and quality analysis",
    research_question="RQ0: composition, quality, duplication and normalization coverage of the full evidence corpus",
    config=dcfg.model_dump(mode="json"), seed=SEED,
    packages=["torch", "numpy", "pandas", "requests", "pydantic", "matplotlib", "cliniver"])
storeB = lambda src: RawStore(RAW, src, f"{SNAP}/{TAG}")
OUTB = pcfg.paths.absolute("data_interim") / SNAP / TAG; OUTB.mkdir(parents=True, exist_ok=True)
PROC = pcfg.paths.absolute("data_processed") / (SNAP + ("_smoke" if SMOKE else "")); PROC.mkdir(parents=True, exist_ok=True)
pmB = PubMedClient(dcfg.pubmed, storeB("pubmed"), W)
ptB = PubTatorClient(dcfg.pubtator, storeB("pubtator"), W)
fdaB = OpenFDAClient(dcfg.openfda, storeB("openfda"), W)
ctB = ClinicalTrialsClient(dcfg.clinicaltrials, storeB("clinicaltrials"), W)
epB = EuropePMCClient(dcfg.europepmc, storeB("europepmc"), W)
YEARS = range(cfgB.pubmed_years[0], cfgB.pubmed_years[1] + 1)
print("MODE:", "SMOKE (not a research result)" if SMOKE else "FULL", "| PubMed years:", YEARS.start, "-", YEARS.stop - 1, "| in-flight caps:",
      {n: c.http.max_in_flight for n, c in [("pubmed", pmB), ("pubtator", ptB), ("openfda", fdaB), ("clinicaltrials", ctB), ("europepmc", epB)]})

## B1. Parallel acquisition
The PMID list is needed by both efetch and PubTator, so those two jobs wait on the `pubmed_ids` job and then run **concurrently** with each other. All other jobs start immediately.

In [ ]:
def b_pubmed_ids():
    per_cond = dict(zip(CONDS, parallel_map(lambda c: pmB.all_ids_by_year(c, YEARS, "B"), CONDS, len(CONDS))))
    membership = defaultdict(set)
    for c, by_year in per_cond.items():
        for ids in by_year.values():
            for pid in ids:
                membership[pid].add(c)
    slice_max = max(len(ids) for by_year in per_cond.values() for ids in by_year.values())
    # [dp] matches both electronic and print publication dates, so an article can appear in two yearly
    # slices: count UNIQUE PMIDs per condition (the sum over slices double-counts those articles).
    per_cond_n = {c: len({p for v in by_year.values() for p in v}) for c, by_year in per_cond.items()}
    in_two = {c: sum(len(v) for v in by_year.values()) - per_cond_n[c] for c, by_year in per_cond.items()}
    return {"pmids": sorted(membership, key=int), "membership": membership, "per_condition": per_cond_n,
            "in_more_than_one_year_slice": in_two, "largest_year_slice": slice_max}


def b_pubmed_docs(f_ids):
    ids = f_ids.result()[1]  # _timed future → (name, result, seconds)
    pmids, B = ids["pmids"], dcfg.pubmed.efetch_batch
    batches = [pmids[i:i + B] for i in range(0, len(pmids), B)]
    def one(ix):
        return [d.model_copy(update={"conditions": sorted(ids["membership"][d.source_id])})
                for d in parse_pubmed_xml(pmB.fetch_xml(batches[ix], f"B_full_{ix:05d}")[0], snapshot=SNAP)]
    return [d for part in parallel_map(one, range(len(batches)), pmB.workers) for d in part]


def b_pubtator(f_ids):
    return ptB.export_parsed(f_ids.result()[1]["pmids"], "B_full")


def b_openfda():
    pairs = dcfg.all_drugs()
    res = parallel_map(lambda cd: fdaB.search_all(cd[1], cfgB.openfda_max_labels_per_drug, rx_only=cfgB.openfda_rx_only,
                                                  cache_prefix="labelB", page_size=min(100, cfgB.openfda_max_labels_per_drug)),
                       pairs, W)
    docs = [parse_label(l, sections=dcfg.openfda.sections, snapshot=SNAP, query_drug=d, condition=c)
            for (c, d), (labels, _) in zip(pairs, res) for l in labels]
    totals = {d: t for (_, d), (_, t) in zip(pairs, res)}
    # A combination label (e.g. sitagliptin + metformin) is returned once per component query drug:
    # merge by doc_id, keeping every query drug and condition (56 duplicates were missed before 2026-11-20).
    qd = defaultdict(set)
    for d in docs:
        qd[d.doc_id].add(d.extra["query_drug"])
    n_fetched = len(docs)
    docs, dup = merge_duplicate_ids(docs)
    docs = [d.model_copy(update={"extra": {**d.extra, "query_drug": min(qd[d.doc_id]), "query_drugs": sorted(qd[d.doc_id])}})
            for d in docs]
    return {"docs": docs, "totals": totals, "fetched": n_fetched, "dup": dup}


def b_clinicaltrials():
    search = (lambda c: ctB.search(c, cfgB.clinicaltrials_page_size)) if SMOKE else \
             (lambda c: ctB.search_all(c, cfgB.clinicaltrials_page_size, "studiesB"))
    res = parallel_map(search, CONDS, len(CONDS))
    docs = [parse_study(s, snapshot=SNAP, condition=c) for c, (studies, _) in zip(CONDS, res) for s in studies]
    docs, dup = merge_duplicate_ids(docs)
    return {"docs": docs, "dup": dup, "totals": {c: t for c, (_, t) in zip(CONDS, res)}}


def b_europepmc():
    strata = [(c, tuple(s)) for c in CONDS for s in cfgB.europepmc_year_strata]
    res = parallel_map(lambda cs: epB.search_oa(cs[0], cfgB.europepmc_per_stratum, years=cs[1],
                                                cache_name=f"searchB_{cs[0]}_{cs[1][0]}"), strata, W)
    metas, seen = [], {}
    for (c, s), (results, hits) in zip(strata, res):
        for m in results:
            if m.get("pmcid") and m["pmcid"] not in seen:
                seen[m["pmcid"]] = (c, s); metas.append((c, s, m))
    def one(csm):
        c, s, meta = csm
        try:
            return parse_jats(epB.full_text_xml(meta["pmcid"]), meta, snapshot=SNAP, condition=c), None
        except (FetchError, KeyError, ET.ParseError) as exc:
            return None, (meta.get("pmcid"), type(exc).__name__, str(exc)[:120])
    out = parallel_map(one, metas, W)
    return {"docs": [d for d, _ in out if d], "fail": [e for _, e in out if e],
            "hits": {f"{c}:{s[0]}-{s[1]}": h for (c, s), (_, h) in zip(strata, res)},
            "stratum_of": {m["pmcid"]: f"{s[0]}-{s[1]}" for c, s, m in metas}}


tB = time.perf_counter()
RB, secB = {}, {}
with ThreadPoolExecutor(max_workers=6) as pool:
    f_ids = pool.submit(_timed, "pubmed_ids", b_pubmed_ids)
    futs = [f_ids, pool.submit(_timed, "pubmed_efetch", lambda: b_pubmed_docs(f_ids)),
            pool.submit(_timed, "pubtator", lambda: b_pubtator(f_ids)),
            pool.submit(_timed, "openfda", b_openfda), pool.submit(_timed, "clinicaltrials", b_clinicaltrials),
            pool.submit(_timed, "europepmc_oa", b_europepmc)]
    for fut in as_completed(futs):
        name, res, dt = fut.result()  # any job's exception is re-raised here
        RB[name], secB[name] = res, dt
wallB = time.perf_counter() - tB
clientsB = {"pubmed": pmB, "pubtator": ptB, "openfda": fdaB, "clinicaltrials": ctB, "europepmc": epB}
reqB = pd.DataFrame({k: {"requests": c.http.n_requests, "retries": c.http.n_retries, "cache_hits": c.http.n_cache_hits}
                     for k, c in clientsB.items()}).T
LIVE_B = int(reqB["requests"].sum()) > 0
expB.log_result("acquisition_timing" if LIVE_B else "replay_timing",
                {"jobs_s": secB, "wall_s": wallB, "requests": reqB.to_dict(orient="index")})
print(f"Stage B {'LIVE acquisition' if LIVE_B else 'REPLAY from raw archive'}: wall {wallB:.1f}s")
display(pd.Series(secB, name="seconds").round(1).sort_values(ascending=False).to_frame()); display(reqB)

## B2. Reconciliation: requested vs received, and caps

In [ ]:
ids = RB["pubmed_ids"]; pm_full = RB["pubmed_efetch"]; ents_B, rels_B, pt_seen = RB["pubtator"]
fdaR, ctR, epR = RB["openfda"], RB["clinicaltrials"], RB["europepmc_oa"]
pm_parsed = {d.source_id for d in pm_full}
recon = {
    "pubmed_ids_unique": len(ids["pmids"]), "pubmed_ids_sum_over_conditions": sum(ids["per_condition"].values()),
    "pubmed_ids_per_condition": ids["per_condition"], "pubmed_ids_in_two_year_slices": ids["in_more_than_one_year_slice"],
    "pubmed_per_condition_equals_population_count": {c: ids["per_condition"][c] == pop_total[c] for c in CONDS},
    "pubmed_largest_year_slice": ids["largest_year_slice"],
    "pubmed_esearch_cap": PubMedClient.ESEARCH_MAX,
    "pubmed_parsed": len(pm_full), "pubmed_ids_without_record": len(set(ids["pmids"]) - pm_parsed),
    "pubmed_population_counts_stageA": pop_total,
    "pubtator_docs_returned": len(pt_seen), "pubtator_missing": len(set(ids["pmids"]) - pt_seen),
    "openfda_labels_fetched": fdaR["fetched"], "openfda_labels_unique": len(fdaR["docs"]),
    "openfda_labels_under_several_query_drugs": fdaR["dup"], "openfda_drugs_without_rx_label": sorted(d for d, t in fdaR["totals"].items() if t == 0),
    "openfda_drugs_truncated_at_limit": {d: t for d, t in fdaR["totals"].items() if t > cfgB.openfda_max_labels_per_drug},
    "clinicaltrials_totals": ctR["totals"], "clinicaltrials_unique": len(ctR["docs"]), "clinicaltrials_cross_condition_dups": ctR["dup"],
    "europepmc_fulltexts": len(epR["docs"]), "europepmc_failures": epR["fail"],
}
expB.log_result("reconciliation", recon)
print(json.dumps(recon, indent=1, default=str))

## B3. Deduplication and exclusions
- **PubMed:** retracted records are excluded (safety rule). Near-duplicate abstracts (≥ 50 words; Jaccard ≥ 0.8) are *annotated* with a group id but **not removed**: they can be legitimately distinct publications (e.g., duplicate publication, or the same abstract in two journals). Whether to collapse them at retrieval time is an NB05 decision.
- **openFDA:** near-duplicate clusters within each (drug, single-ingredient vs combination) group are collapsed to the most recent label. The other labels' set-ids are kept as aliases (D-NB01-3).

In [ ]:
t0 = time.perf_counter()
retracted = [d for d in pm_full if d.extra.get("is_retracted")]
pm_keep = [d for d in pm_full if not (cfgB.exclude_retracted and d.extra.get("is_retracted"))]
long_pm = [d for d in pm_keep if len(d.text.split()) >= cfgB.near_dup_min_words]
pm_pairs, pm_clusters = near_duplicate_clusters([d.doc_id for d in long_pm], [d.text for d in long_pm],
                                                threshold=cfgB.near_dup_threshold, seed=SEED)
group_of = {doc_id: f"pmdup{gi:05d}" for gi, cl in enumerate(pm_clusters) for doc_id in cl}
pm_keep = [d.model_copy(update={"extra": {**d.extra, "near_dup_group": group_of[d.doc_id]}}) if d.doc_id in group_of else d
           for d in pm_keep]
t_pm = time.perf_counter() - t0

t0 = time.perf_counter()
fda_reps, fda_aliases, fda_cluster_sizes = [], {}, []
groups = defaultdict(list)
for d in fdaR["docs"]:
    groups[(d.extra["query_drug"], d.extra["is_combination"])].append(d)
for key, members in groups.items():
    _, clusters = near_duplicate_clusters([d.doc_id for d in members], [d.text for d in members],
                                          threshold=cfgB.near_dup_threshold, seed=SEED)
    in_cluster = set()
    by_id = {d.doc_id: d for d in members}
    for cl in clusters:
        docs_cl = sorted((by_id[i] for i in cl), key=lambda d: (d.pub_date or date.min, d.doc_id), reverse=True)
        rep = docs_cl[0]
        fda_aliases[rep.doc_id] = [x.identifiers["set_id"] for x in docs_cl[1:]]
        fda_reps.append(rep.model_copy(update={"extra": {**rep.extra, "n_near_duplicate_labels": len(docs_cl) - 1}}))
        in_cluster |= set(cl); fda_cluster_sizes.append(len(cl))
    fda_reps += [d for d in members if d.doc_id not in in_cluster]
t_fda = time.perf_counter() - t0

dedup = {
    "pubmed_retracted_excluded": len(retracted), "pubmed_after_exclusion": len(pm_keep),
    "pubmed_near_dup_checked_docs": len(long_pm), "pubmed_lsh_candidate_pairs": pm_pairs.attrs.get("n_candidates"),
    "pubmed_exact_checks": pm_pairs.attrs.get("n_exact_checks"),
    "pubmed_near_dup_edges_verified": len(pm_pairs), "pubmed_near_dup_clusters": len(pm_clusters),
    "pubmed_docs_in_near_dup_clusters": sum(len(c) for c in pm_clusters), "pubmed_near_dup_seconds": round(t_pm, 1),
    "openfda_labels_before": len(fdaR["docs"]), "openfda_labels_after": len(fda_reps),
    "openfda_clusters": len(fda_cluster_sizes), "openfda_largest_cluster": max(fda_cluster_sizes, default=0),
    "openfda_dedup_seconds": round(t_fda, 1),
}
expB.log_result("deduplication", dedup)
expB.save_table("pubmed_near_dup_pairs", pm_pairs)
print(json.dumps(dedup, indent=1))

In [ ]:
# Inspect a few PubMed near-duplicate pairs: what kind of duplication is this?
show = pm_pairs.sort_values("jaccard", ascending=False).head(6)
by_id = {d.doc_id: d for d in pm_keep}
for _, r in show.iterrows():
    a, b = by_id[r.doc_a], by_id[r.doc_b]
    print(f"J={r.jaccard:.2f} | {a.doc_id} ({a.pub_date}, {a.journal}) vs {b.doc_id} ({b.pub_date}, {b.journal})")
    print("   A:", a.title[:100]); print("   B:", b.title[:100])

## B4. Assemble and persist the corpus

In [ ]:
ct_full, ep_full = ctR["docs"], [d.model_copy(update={"extra": {**d.extra, "year_stratum": epR["stratum_of"][d.source_id]}}) for d in epR["docs"]]
corpus = pm_keep + fda_reps + ct_full + ep_full + who_docs
countsB = {
    "documents_interim_all": write_jsonl(pm_full + fdaR["docs"] + ct_full + ep_full + who_docs, OUTB / "documents_all.jsonl"),
    "documents": write_jsonl(corpus, PROC / "documents.jsonl"),
    "concepts": write_jsonl(concepts, PROC / "concepts.jsonl"),
    "pubtator_entities": write_jsonl(ents_B, PROC / "pubtator_entities.jsonl"),
    "pubtator_relations": write_jsonl(rels_B, PROC / "pubtator_relations.jsonl"),
}
(PROC / "exclusions_and_aliases.json").write_text(json.dumps({
    "retracted_excluded_pmids": sorted(d.source_id for d in retracted),
    "openfda_aliases": fda_aliases, "pubmed_near_dup_clusters": pm_clusters}, indent=1), encoding="utf-8")
sizes = {p.name: round(p.stat().st_size / 1e6, 1) for p in PROC.glob("*.jsonl")}
manifestB = {"snapshot": SNAP, "stage": "B", "created_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
             "counts": countsB, "by_source": dict(Counter(d.source for d in corpus)), "file_mb": sizes,
             "licenses": dict(Counter(d.license for d in corpus).most_common(12)),
             "source_snapshot_dates": {"who_gho": SNAP + " (Stage A fetch)", "concepts": SNAP + " (Stage A fetch)",
                                       "others": SNAP + " (Stage B fetch)"}}
(PROC / "manifest.json").write_text(json.dumps(manifestB, indent=1), encoding="utf-8")
expB.log_result("corpus_manifest", manifestB)
print(json.dumps(manifestB, indent=1))

## B5. Evaluation: full-corpus quality
### B5.1 Composition and completeness

In [ ]:
dfB = docs_to_frame(corpus)
compB = composition(dfB); complB = completeness(corpus)
expB.save_table("composition", compB); expB.save_table("completeness", complB)
expB.log_result("composition", compB.to_dict(orient="index")); expB.log_result("completeness", complB.round(3).to_dict(orient="index"))
display(compB); display(complB.round(3))

### B5.2 Evidence types in the full PubMed corpus (actual), vs Stage A population upper bounds

In [ ]:
pmB_df = dfB[dfB.source == "pubmed"].copy()
pmB_df["stratum"] = pmB_df.publication_types.map(evidence_stratum)
actual = pmB_df.stratum.value_counts().rename("corpus_docs")
evB = pd.concat([actual, (actual / actual.sum()).round(3).rename("corpus_share"), pop_share.rename("population_upper_bound_share").round(3)], axis=1)
evB["corpus_docs"] = evB["corpus_docs"].fillna(0).astype(int)  # population share stays NaN where no population query exists ("other")
by_cond = (pmB_df.explode("conditions").groupby(["conditions", "stratum"]).size().unstack(fill_value=0))
expB.save_table("evidence_strata", evB); expB.save_table("evidence_strata_by_condition", by_cond)
expB.log_result("evidence_strata", evB.astype(object).where(evB.notna(), None).to_dict(orient="index"))  # NaN → null (valid JSON)
display(evB); display(by_cond)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.5))
evB[["corpus_share", "population_upper_bound_share"]].sort_values("corpus_share").plot.barh(ax=ax)
ax.set_xlabel("share of PubMed documents"); ax.set_title("Evidence strata: full corpus (exclusive, highest tier first) vs population upper bounds")
expB.save_figure("evidence_strata", fig); plt.show()

### B5.3 Temporal distribution and quality flags

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.5))
for s, g in dfB[dfB.year.notna() & (dfB.source != "who_gho")].groupby("source"):
    ax.hist(g.year, bins=range(1990, 2028), alpha=.5, label=f"{s} (n={len(g)})")
ax.set_yscale("log"); ax.set_ylabel("documents (log)"); ax.legend(fontsize=7); ax.set_title("Full corpus: document year by source")
expB.save_figure("temporal", fig); plt.show()
qB = quality_flags(pm_full + fdaR["docs"] + ct_full + ep_full)  # before exclusions, so retractions are counted
expB.log_result("quality_flags", qB); pd.Series(qB, name="value").to_frame()

### B5.4 PubTator coverage, and cross-source links

In [ ]:
drug_names = {d.lower() for _, d in dcfg.all_drugs()} | {s.lower() for x in concepts if x.source == "rxnorm" for s in x.synonyms}
ptB_cov = pubtator_coverage(ents_B, rels_B, pm_keep, drug_names)
pm_ids_set = {d.source_id for d in pm_keep}
links = {
    "trials_with_result_pmids": sum(bool(d.identifiers.get("result_pmids")) for d in ct_full),
    "trials_with_result_pmid_in_corpus": sum(bool(set(d.identifiers.get("result_pmids", [])) & pm_ids_set) for d in ct_full),
    "europepmc_oa_with_pmid_in_corpus": sum(d.identifiers.get("pmid") in pm_ids_set for d in ep_full),
}
expB.log_result("pubtator_coverage", ptB_cov); expB.log_result("cross_source_links", links)
print(json.dumps(ptB_cov, indent=1)); print(json.dumps(links, indent=1))

### B5.5 Europe PMC long-document subset (input to NB02)

In [ ]:
epdf = pd.DataFrame([{"pmcid": d.source_id, "stratum": d.extra["year_stratum"], "words": len(d.text.split()),
                      "sections": len(d.sections), "tables": d.extra["n_tables"], "license": d.license} for d in ep_full])
ep_summary = epdf.groupby("stratum").agg(docs=("pmcid", "count"), words_median=("words", "median"), sections_median=("sections", "median"), tables=("tables", "sum"))
expB.save_table("europepmc_subset", ep_summary); expB.log_result("europepmc_subset", ep_summary.to_dict(orient="index"))
display(ep_summary); display(epdf.license.value_counts().to_frame())

In [ ]:
expB.finish()
print("results:", expB.dir)

## B6. Discussion (Stage B)
All figures are copied from the outputs above (`artifacts/experiments/EXP-01-003/results.json`). Where a count refers to a set other than the final corpus, that set is named.

### B6.1 Completeness of acquisition (requested vs received)
| Source | Requested / available | Received | Notes |
|---|---|---|---|
| PubMed | 5 condition populations | **125,022 unique PMIDs, 125,022 parsed (0 missing)** | Per-condition unique counts equal the Stage A population counts **exactly** for all 5 conditions. Largest yearly slice 3,386 < esearch cap 9,999, so no slice was truncated |
| PubTator 3 | 125,022 PMIDs | 124,814 (99.8%) | 208 PMIDs not available from PubTator; recorded |
| openFDA | 49 drugs, Rx labels | 4,514 fetched → **4,114 unique** | All 49 drugs have ≥ 1 Rx label; none hit the 1,000-label cap. 400 fetches were the same combination label returned for a second component drug (merged) |
| ClinicalTrials.gov | completed drug trials, 5 conditions | **9,665 unique** | API totals reconciled per condition (4,177 / 2,971 / 1,360 / 537 / 1,129); 509 trials registered under > 1 condition (merged) |
| Europe PMC OA | 4 year strata × ~100 | **398 full texts** | 99–100 per stratum |
| WHO GHO | Stage A statements | 3,444 | Re-used from the Stage A fetch of the same snapshot day |

**Final corpus (snapshot 2026-11-20): 138,764 documents** (PubMed 124,918; ClinicalTrials.gov 9,665; WHO 3,444; Europe PMC 398; openFDA 339), plus 190 normalization concepts, 3,020,745 PubTator entity mentions and 386,627 PubTator relations.

### B6.2 Exclusions and duplicates
- **Retractions:** 104 retracted PubMed articles excluded (IDs listed in `exclusions_and_aliases.json`). A further **2,330** articles carry a correction or expression-of-concern link; they are kept but flagged, so a verdict citing them can say so.
- **PubMed near-duplicates:** 261 abstracts in 125 clusters (0.2% of the 123,583 abstracts with ≥ 50 words). The inspected top pairs are the same article indexed twice under two PMIDs (e.g. journal-supplement articles re-indexed in a later year). They are **annotated** (`near_dup_group`), not removed (ADR-0008), so NB05 can collapse them at ranking time.
- **openFDA near-duplicates:** the 4,114 unique labels collapse to **339** (200 clusters, largest 207 labels of one drug). About 92% of unique labels are near-identical copies (repackagers and generic manufacturers). The representative is the most recent label; the other set-ids are kept as aliases. Clustering is done **within** one drug (and combination flag) only, so no cluster mixes drugs.
- **Cross-condition overlap** is real and expected: 130,261 condition memberships vs 125,022 unique PMIDs.

### B6.3 What evidence the corpus contains
| Stratum (exclusive, highest tier first) | Corpus docs | Corpus share | Population upper bound (Stage A) |
|---|---|---|---|
| Meta-analysis | 5,418 | 4.3% | 4.8% |
| Systematic review | 1,730 | 1.4% | 4.0% |
| RCT | 22,339 | 17.9% | 17.9% |
| Other clinical trial | 5,306 | 4.2% | 4.2% |
| Observational | 4,526 | 3.6% | 3.6% |
| Guideline | 306 | 0.2% | 0.3% |
| Review | 29,096 | 23.3% | 25.8% |
| Case report | 4,576 | 3.7% | 4.1% |
| Other | 51,621 | 41.3% | — (no population query) |

- The full-population corpus reproduces the population evidence mix (RCT 17.9% vs 17.9%), unlike the Stage A Best Match sample (RCT 0.6%). This confirms D-NB01-1.
- Shares below the upper bound are expected, because the strata are exclusive. For example, an article tagged both systematic review and meta-analysis is counted once, as a meta-analysis, which is why the systematic-review share (1.4%) is well below its upper bound (4.0%).
- **Guidelines are scarce (306).** Claims phrased as recommendations ("first-line", "recommended") will often lack guideline-level support in this corpus. Copyrighted society guidelines are excluded (D2), so INSUFFICIENT EVIDENCE is the correct outcome there, not a failure.

### B6.4 Quality and structure
- **PubMed:** 51% structured abstracts; median 251 words; 1,207 abstracts have < 50 words (1.0%); 34 English abstracts recovered from `OtherAbstract`; 0 empty. DOI present for 93.1%; day-precision dates for 62.7% (the rest are month or year only, so evidence dates must carry their precision).
- **openFDA** (final corpus: median 7,078 words, 13 sections): long, sectioned documents. Section-aware chunking is mandatory in NB02. Among the 4,114 unique labels:
  - 678 are combination products.
  - 48 lack at least one key section.
  - 22 have no named sections and use the `spl_unclassified_section` fallback.
  - 1 label (SOAANZ, torsemide) has **no clinical text in openFDA at all**: only product data and package-panel image captions. This is a gap in the source, not a parser defect, and NB02 drops empty documents explicitly.
- **ClinicalTrials.gov:**
  - 3,300 trials (34.1%) have posted results.
  - 5,036 (52.1%) have no detailed description, so registry text is short (median 97 words) and mostly describes design, not outcomes.
  - 1,135 trials cite a result publication; for **797 (70.2%)** that PMID is in the PubMed corpus. This is a usable trial→publication link.
- **Europe PMC OA:** median 4,169 words and 8 sections, balanced across strata (99–100 each). There are **1,057 tables that are not in the extracted text**: NB02 must measure that loss or handle tables separately. 132 of 398 articles are also in the PubMed corpus (abstract and full text of the same study, useful for the abstract-only limitation analysis).
- **PubTator coverage** (over the 124,918 corpus PubMed documents):
  - any entity 99.3%, Disease 96.6%, Chemical 77.7%, Chemical and Disease 75.6%; 81% of chemical mentions are normalized to an identifier.
  - Exact in-scope drug-name mentions: 38,557 (30.9%), a lower bound.
  - Relations are dominated by coarse statistical types: Negative_Correlation 214,721, Association 84,905, Positive_Correlation 60,324. Drug_Interaction has only 59 relations, so **PubTator cannot serve as an interaction source**, and graph edges remain retrieval signals, never evidence (ADR-0003).

### B6.5 Error analysis: defects found and fixed during Stage B
Each defect was traced to its root cause, fixed in `src/cliniver/`, and (except for item 3, in the runner script) covered by a regression test (50 unit tests pass).
1. **Hung PubMed connections:** a server trickling bytes never trips a per-read timeout. Fix: a total request deadline enforced by a watchdog. The first version deadlocked on Windows (`Response.close` blocks), so the watchdog shuts the socket down instead.
2. **Quadratic near-duplicate verification:** near-identical openFDA labels formed cliques (~530k exact comparisons, > 1 h). Fix: skip candidate pairs already in the same component. This gives identical clusters (tested against brute force), and openFDA dedup now takes about 98 s.
3. **Runner masked a timeout:** the notebook runner reported success after a cell timeout. Fix: any exception or error output sets status 1.
4. **Per-condition counts double-counted** articles that `[dp]` places in two yearly slices. Fix: unique counts per condition, which now equal the population counts exactly.
5. **Combination labels counted under each component drug** (400 duplicates). Fix: merge by label ID before deduplication.
6. **Empty documents:**
   - SPL labels without named sections (now use `spl_unclassified_section`).
   - JATS articles with paragraphs directly under `<body>` (5 previously empty; now 0).

### B6.6 Acquisition cost (live run, 2026-11-20)
- The live run took 2,902 s wall time. The critical path was Europe PMC full text (2,902 s, 117 retries against slow responses). PubMed efetch took 665 s and PubTator 478 s, with **0 retries** for both (the EXP-01-002 pacing held).
- A replay from the raw archive makes **0 network requests**: about 2 min for the archive and about 5.5 min for the whole notebook.

## B7. Limitations (Stage B)
- **Indexing lag:** recent articles (2025–2026) are under-represented, because MeSH indexing lags publication.
- **Abstracts, not full text:** PubMed abstracts can omit qualifiers. Full text covers only the 398-article Europe PMC subset.
- **openFDA:**
  - US labels only. Label dates are revision dates, not evidence dates.
  - Collapsing near-duplicates keeps one representative, so small differences between aliased labels (e.g. strengths, inactive ingredients) are not searchable.
- **Registry records** describe design more than outcomes.
- **WHO statistics** describe populations and cannot verify individual-level claims.
- **One snapshot day:** server-side changes after 2026-11-20 are not reflected. Every verdict must state the snapshot date (ADR-0005).
- **Evidence strata** come from PubMed publication types, which are assigned with a lag and can be incomplete. "Other" (41.3%) is not further classified.
- **Missing annotations:** 208 PubMed documents (0.2%) have no PubTator annotations, so they are invisible to the graph channel but not to BM25 or dense retrieval.

## B8. Conclusion and decision
**Conclusion:**
- The corpus for snapshot **2026-11-20** is complete with respect to its defined populations: every requested PubMed record was received, and the per-condition counts match the population counts exactly. It is deduplicated, with retractions excluded and corrections flagged, and fully reproducible from the raw archive.
- It reproduces the population evidence mix, which the Stage A sample did not.
- Its main structural risks for later notebooks are long sectioned labels, tables missing from full text, scarce guidelines, and coarse PubTator relations.

**Decision (proposed, pending owner approval):** freeze `data/processed/2026-11-20/` as the input to NB02–NB08.

**Next steps (NB02, document processing and chunking):**
1. Compare chunking strategies:
   - by structured-abstract section for PubMed;
   - by SPL section for openFDA;
   - long-document strategies on the Europe PMC subset.
2. Drop empty-text documents explicitly.
3. Measure what is lost with tables.